In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel


def get_inputs(pairs, tokenizer, prompt=None, max_length=1024):
    if prompt is None:
        prompt =  "Analyze the medical condition described in query A and the statement in passage B. Determine if the statement is in accordance with the condition. Respond with 'Yes' if they are suitable and aligned or 'No' if otherwise."
    sep = "\n"
    prompt_inputs = tokenizer(prompt,
                              return_tensors=None,
                              add_special_tokens=False)['input_ids']
    sep_inputs = tokenizer(sep,
                           return_tensors=None,
                           add_special_tokens=False)['input_ids']
    inputs = []
    for query, passage in pairs:
        query_inputs = tokenizer(f'A: {query}',
                                 return_tensors=None,
                                 add_special_tokens=False,
                                 max_length=max_length * 3 // 4,
                                 truncation=True)
        passage_inputs = tokenizer(f'B: {passage}',
                                   return_tensors=None,
                                   add_special_tokens=False,
                                   max_length=max_length,
                                   truncation=True)
        item = tokenizer.prepare_for_model(
            [tokenizer.bos_token_id] + query_inputs['input_ids'],
            sep_inputs + passage_inputs['input_ids'],
            truncation='only_second',
            max_length=max_length,
            padding=False,
            return_attention_mask=False,
            return_token_type_ids=False,
            add_special_tokens=False
        )
        item['input_ids'] = item['input_ids'] + sep_inputs + prompt_inputs
        item['attention_mask'] = [1] * len(item['input_ids'])
        inputs.append(item)
    return tokenizer.pad(
            inputs,
            padding=True,
            max_length=max_length + len(sep_inputs) + len(prompt_inputs),
            pad_to_multiple_of=8,
            return_tensors='pt',
    )

tokenizer = AutoTokenizer.from_pretrained('BAAI/bge-reranker-v2-gemma')
model = AutoModelForCausalLM.from_pretrained('BAAI/bge-reranker-v2-gemma')

# Path to your local PEFT adapter directory
adapter_path = "/home/mabdallah/TrialMatchAI/src/Matcher/finetuned_reranker/"

# Load the PEFT adapter
model = PeftModel.from_pretrained(model, adapter_path)

yes_loc = tokenizer('Yes', add_special_tokens=False)['input_ids'][0]
model.eval()

pairs = [['Patient with KRAS mutation', 'Patient without KRAS mutation'], ['what is panda?', 'The giant panda (Ailuropoda melanoleuca), sometimes called a panda bear or simply panda, is a bear species endemic to China.']]*1000
with torch.no_grad():
    inputs = get_inputs(pairs, tokenizer)
    scores = model(**inputs, return_dict=True).logits[:, -1, yes_loc].view(-1, ).float()
    print(scores)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from typing import List, Optional, Tuple, Dict


class Reranker:
    def __init__(
        self,
        base_model: str,
        adapter_path: str,
        prompt: str = None,
        max_length: int = 1024,
        device: str = None
    ):
        """
        Initializes the Reranker with the specified base model, adapter, and tokenizer.

        Args:
            base_model (str): The name or path of the pretrained base model.
            adapter_path (str): The path to the local PEFT adapter directory.
            prompt (Optional[str]): The prompt to use for analysis. If None, a default prompt is used.
            max_length (int): The maximum length for tokenization.
            device (Optional[str]): The device to run the model on (e.g., 'cpu', 'cuda'). If None, defaults to CUDA if available.
        
        Raises:
            ValueError: If the base model or adapter cannot be loaded.
        """
        self.base_model = base_model
        self.adapter_path = adapter_path
        self.max_length = max_length

        # Initialize the tokenizer
        try:
            self.tokenizer = AutoTokenizer.from_pretrained(self.base_model)
        except Exception as e:
            raise ValueError(f"Failed to load tokenizer for base model '{self.base_model}': {e}")

        # Initialize the base model
        try:
            self.model = AutoModelForCausalLM.from_pretrained(self.base_model)
        except Exception as e:
            raise ValueError(f"Failed to load base model '{self.base_model}': {e}")

        # Load the PEFT adapter
        try:
            self.model = PeftModel.from_pretrained(self.model, self.adapter_path)
        except Exception as e:
            raise ValueError(f"Failed to load PEFT adapter from '{self.adapter_path}': {e}")

        self.model.eval()

        # Determine the device
        if device:
            self.device = device
        else:
            self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.model.to(self.device)

        # Define the prompt
        if prompt is None:
            self.prompt = (
                "Analyze the medical condition described in query A and the statement in passage B. "
                "Determine if the statement is in accordance with the condition. "
                "Respond with 'Yes' if they are suitable and aligned or 'No' if otherwise."
            )
        else:
            self.prompt = prompt

        # Precompute token IDs for efficiency
        self.sep_token = "\n"
        self.bos_token_id = self.tokenizer.bos_token_id
        yes_token_ids = self.tokenizer('Yes', add_special_tokens=False)['input_ids']
        if not yes_token_ids:
            raise ValueError("The tokenizer could not find the token ID for 'Yes'.")
        self.yes_token_id = yes_token_ids[0]

    def get_inputs(
        self,
        pairs: List[Tuple[str, str]]
    ) -> Dict[str, torch.Tensor]:
        """
        Prepares the input tensors for the model based on the input pairs.

        Args:
            pairs (List[Tuple[str, str]]): A list of tuples where each tuple contains a query and a passage.

        Returns:
            Dict[str, torch.Tensor]: A dictionary containing the input_ids and attention_mask tensors.
        """
        sep = self.sep_token
        prompt_inputs = self.tokenizer(
            self.prompt,
            return_tensors=None,
            add_special_tokens=False
        )['input_ids']
        sep_inputs = self.tokenizer(
            sep,
            return_tensors=None,
            add_special_tokens=False
        )['input_ids']

        inputs = []
        for query, passage in pairs:
            query_inputs = self.tokenizer(
                f'A: {query}',
                return_tensors=None,
                add_special_tokens=False,
                max_length=self.max_length * 3 // 4,
                truncation=True
            )
            passage_inputs = self.tokenizer(
                f'B: {passage}',
                return_tensors=None,
                add_special_tokens=False,
                max_length=self.max_length,
                truncation=True
            )
            item = self.tokenizer.prepare_for_model(
                [self.bos_token_id] + query_inputs['input_ids'],
                sep_inputs + passage_inputs['input_ids'],
                truncation='only_second',
                max_length=self.max_length,
                padding=False,
                return_attention_mask=False,
                return_token_type_ids=False,
                add_special_tokens=False
            )
            item['input_ids'] = item['input_ids'] + sep_inputs + prompt_inputs
            item['attention_mask'] = [1] * len(item['input_ids'])
            inputs.append(item)

        padded = self.tokenizer.pad(
            inputs,
            padding=True,
            max_length=self.max_length + len(sep_inputs) + len(prompt_inputs),
            pad_to_multiple_of=8,
            return_tensors='pt',
        )

        # Move tensors to the specified device
        return {k: v.to(self.device) for k, v in padded.items()}

    def score_pairs(
        self,
        pairs: List[Tuple[str, str]]
    ) -> torch.Tensor:
        """
        Computes the scores for each pair indicating the alignment based on the model's logits.

        Args:
            pairs (List[Tuple[str, str]]): A list of tuples where each tuple contains a query and a passage.

        Returns:
            torch.Tensor: A tensor containing the scores for each pair.
        """
        with torch.no_grad():
            inputs = self.get_inputs(pairs)
            outputs = self.model(**inputs, return_dict=True)
            logits = outputs.logits  # Shape: (batch_size, sequence_length, vocab_size)
            # Extract logits for the last token
            last_token_logits = logits[:, -1, :]  # Shape: (batch_size, vocab_size)
            # Get the logits corresponding to the 'Yes' token
            yes_logits = last_token_logits[:, self.yes_token_id]
            return yes_logits.view(-1).float()

    def compute_scores(self, pairs: List[Tuple[str, str]]) -> List[float]:
        """
        Analyzes the given pairs and returns their corresponding scores.

        Args:
            pairs (List[Tuple[str, str]]): A list of tuples where each tuple contains a query and a passage.

        Returns:
            List[float]: A list of scores indicating the alignment for each pair.
        """
        scores = self.score_pairs(pairs)
        return scores.cpu().tolist()


# Example Usage
if __name__ == "__main__":
    # Specify the base model and adapter path
    base_model = 'BAAI/bge-reranker-v2-gemma'
    adapter_path = "/home/mabdallah/TrialMatchAI/src/Matcher/finetuned_reranker/"

    # Initialize the Reranker with specified base model and adapter path
    reranker = Reranker(
        base_model=base_model,
        adapter_path=adapter_path
    )

    # Define the pairs to analyze
    pairs = [
        ('Patient with KRAS mutation', 'Patient without KRAS mutation'),
        (
            'What is a panda?',
            'The giant panda (Ailuropoda melanoleuca), sometimes called a panda bear or simply panda, is a bear species endemic to China.'
        )
    ]

    # Analyze the pairs and print the scores
    scores = reranker.compute_scores(pairs)
    print(scores)


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

class Reranker:
    def __init__(self, model_name, adapter_path, max_length=1024, batch_size=8):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(model_name)
        self.model = PeftModel.from_pretrained(self.model, adapter_path)
        self.model.eval()
        self.yes_loc = self.tokenizer('Yes', add_special_tokens=False)['input_ids'][0]
        self.max_length = max_length
        self.batch_size = batch_size

    def get_inputs(self, pairs, prompt=None):
        if prompt is None:
            prompt = "Given a query A and a passage B, determine whether the passage contains an answer to the query by providing a prediction of either 'Yes' or 'No'."
        sep = "\n"
        prompt_inputs = self.tokenizer(prompt,
                                       return_tensors=None,
                                       add_special_tokens=False)['input_ids']
        sep_inputs = self.tokenizer(sep,
                                    return_tensors=None,
                                    add_special_tokens=False)['input_ids']
        inputs = []
        for query, passage in pairs:
            query_inputs = self.tokenizer(f'A: {query}',
                                          return_tensors=None,
                                          add_special_tokens=False,
                                          max_length=self.max_length * 3 // 4,
                                          truncation=True)
            passage_inputs = self.tokenizer(f'B: {passage}',
                                            return_tensors=None,
                                            add_special_tokens=False,
                                            max_length=self.max_length,
                                            truncation=True)
            item = self.tokenizer.prepare_for_model(
                [self.tokenizer.bos_token_id] + query_inputs['input_ids'],
                sep_inputs + passage_inputs['input_ids'],
                truncation='only_second',
                max_length=self.max_length,
                padding=False,
                return_attention_mask=False,
                return_token_type_ids=False,
                add_special_tokens=False
            )
            item['input_ids'] = item['input_ids'] + sep_inputs + prompt_inputs
            item['attention_mask'] = [1] * len(item['input_ids'])
            inputs.append(item)
        return self.tokenizer.pad(
            inputs,
            padding=True,
            max_length=self.max_length + len(sep_inputs) + len(prompt_inputs),
            pad_to_multiple_of=8,
            return_tensors='pt',
        )

    def rerank(self, pairs, prompt=None):
        scores = []
        for i in range(0, len(pairs), self.batch_size):
            batch_pairs = pairs[i:i + self.batch_size]
            inputs = self.get_inputs(batch_pairs, prompt)
            with torch.no_grad():
                logits = self.model(**inputs, return_dict=True).logits
                batch_scores = logits[:, -1, self.yes_loc].view(-1).float().tolist()
                scores.extend(batch_scores)
        return scores

# Example Usage
model_name = 'BAAI/bge-reranker-v2-gemma'
adapter_path = "/home/mabdallah/TrialMatchAI/src/Matcher/finetuned_reranker/"
reranker = Reranker(model_name, adapter_path, batch_size=5)

pairs = [['what is panda?', 'hi'], ['what is panda?', 'The giant panda (Ailuropoda melanoleuca), sometimes called a panda bear or simply panda, is a bear species endemic to China.']]*50
scores = reranker.rerank(pairs)
print(scores)


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

class Reranker:
    def __init__(self, model_name, adapter_path, max_length=1024, batch_size=8):
        # Check if GPU is available and set the device
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        # Load tokenizer and model
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(model_name)
        
        # Load PEFT adapter and move model to device
        self.model = PeftModel.from_pretrained(self.model, adapter_path)
        self.model.to(self.device)  # Move model to GPU if available
        self.model.eval()
        
        # Tokenize "Yes" for later use and set other parameters
        self.yes_loc = self.tokenizer('Yes', add_special_tokens=False)['input_ids'][0]
        self.max_length = max_length
        self.batch_size = batch_size

    def get_inputs(self, pairs, prompt=None):
        if prompt is None:
            prompt = "Given a query A and a passage B, determine whether the passage contains an answer to the query by providing a prediction of either 'Yes' or 'No'."
        
        # Tokenizing the prompt and separator
        sep = "\n"
        prompt_inputs = self.tokenizer(prompt,
                                       return_tensors=None,
                                       add_special_tokens=False)['input_ids']
        sep_inputs = self.tokenizer(sep,
                                    return_tensors=None,
                                    add_special_tokens=False)['input_ids']
        inputs = []
        
        # Tokenizing each pair of query and passage
        for query, passage in pairs:
            query_inputs = self.tokenizer(f'A: {query}',
                                          return_tensors=None,
                                          add_special_tokens=False,
                                          max_length=self.max_length * 3 // 4,
                                          truncation=True)
            passage_inputs = self.tokenizer(f'B: {passage}',
                                            return_tensors=None,
                                            add_special_tokens=False,
                                            max_length=self.max_length,
                                            truncation=True)
            # Prepare input with truncation on the second part
            item = self.tokenizer.prepare_for_model(
                [self.tokenizer.bos_token_id] + query_inputs['input_ids'],
                sep_inputs + passage_inputs['input_ids'],
                truncation='only_second',
                max_length=self.max_length,
                padding=False,
                return_attention_mask=False,
                return_token_type_ids=False,
                add_special_tokens=False
            )
            # Final input with prompt and separator
            item['input_ids'] = item['input_ids'] + sep_inputs + prompt_inputs
            item['attention_mask'] = [1] * len(item['input_ids'])
            inputs.append(item)
        
        # Padding the inputs and converting them to PyTorch tensors
        padded_inputs = self.tokenizer.pad(
            inputs,
            padding=True,
            max_length=self.max_length + len(sep_inputs) + len(prompt_inputs),
            pad_to_multiple_of=8,
            return_tensors='pt',
        )
        
        # Move input tensors to the device (GPU or CPU)
        padded_inputs = {key: tensor.to(self.device) for key, tensor in padded_inputs.items()}
        return padded_inputs

    def rerank(self, pairs, prompt=None):
        scores = []
        # Processing the pairs in batches to avoid memory overload
        for i in range(0, len(pairs), self.batch_size):
            batch_pairs = pairs[i:i + self.batch_size]
            inputs = self.get_inputs(batch_pairs, prompt)
            with torch.no_grad():
                # Run the model and compute logits on the device
                logits = self.model(**inputs, return_dict=True).logits
                batch_scores = logits[:, -1, self.yes_loc].view(-1).float().tolist()
                scores.extend(batch_scores)
        return scores

# Example Usage
model_name = 'BAAI/bge-reranker-v2-gemma'
adapter_path = "/home/mabdallah/TrialMatchAI/src/Matcher/finetuned_reranker/"
reranker = Reranker(model_name, adapter_path, batch_size=3)

pairs = [['what is panda?', 'hi'], ['what is panda?', 'The giant panda (Ailuropoda melanoleuca), sometimes called a panda bear or simply panda, is a bear species endemic to China.']]*50
scores = reranker.rerank(pairs)
print(scores)


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

class Reranker:
    def __init__(self, model_name, adapter_path, max_length=1024, batch_size=8):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(model_name)
        self.model = PeftModel.from_pretrained(self.model, adapter_path)
        self.model.to(self.device)
        self.model.eval()
        self.yes_loc = self.tokenizer('Yes', add_special_tokens=False)['input_ids'][0]
        self.max_length = max_length
        self.batch_size = batch_size

    def get_inputs(self, pairs, prompt=None):
        if prompt is None:
            prompt = "Given a query A and a passage B, determine whether the passage contains an answer to the query by providing a prediction of either 'Yes' or 'No'."
        
        # Tokenize each query-passage pair individually
        inputs = []
        sep = "\n"
        prompt_inputs = self.tokenizer(prompt, return_tensors=None, add_special_tokens=False)['input_ids']
        sep_inputs = self.tokenizer(sep, return_tensors=None, add_special_tokens=False)['input_ids']
        
        for query, passage in pairs:
            query_inputs = self.tokenizer(f'A: {query}',
                                          return_tensors=None,
                                          add_special_tokens=False,
                                          max_length=self.max_length * 3 // 4,
                                          truncation=True)
            passage_inputs = self.tokenizer(f'B: {passage}',
                                            return_tensors=None,
                                            add_special_tokens=False,
                                            max_length=self.max_length,
                                            truncation=True)
            # Combine query and passage into the final input with separator and prompt
            item = self.tokenizer.prepare_for_model(
                [self.tokenizer.bos_token_id] + query_inputs['input_ids'],
                sep_inputs + passage_inputs['input_ids'],
                truncation='only_second',
                max_length=self.max_length,
                padding=False,
                return_attention_mask=False,
                return_token_type_ids=False,
                add_special_tokens=False
            )
            item['input_ids'] = item['input_ids'] + sep_inputs + prompt_inputs
            item['attention_mask'] = [1] * len(item['input_ids'])
            inputs.append(item)

        # Padding the inputs and converting them to PyTorch tensors
        padded_inputs = self.tokenizer.pad(
            inputs,
            padding=True,
            max_length=self.max_length + len(sep_inputs) + len(prompt_inputs),
            pad_to_multiple_of=8,
            return_tensors='pt'
        )
        # Move input tensors to the device (GPU or CPU)
        padded_inputs = {key: tensor.to(self.device) for key, tensor in padded_inputs.items()}
        return padded_inputs

    def rerank(self, pairs, prompt=None):
        scores = []
        # Process the pairs in batches
        for i in range(0, len(pairs), self.batch_size):
            batch_pairs = pairs[i:i + self.batch_size]
            inputs = self.get_inputs(batch_pairs, prompt)
            
            with torch.no_grad():
                # Run the model and compute logits on the device
                logits = self.model(**inputs, return_dict=True).logits

                # Extract the logits corresponding to the final token of each example
                for logit in logits:
                    yes_score = logit[-1, self.yes_loc].item()
                    scores.append(yes_score)

        return scores

# Example Usage
model_name = 'BAAI/bge-reranker-v2-gemma'
adapter_path = "/home/mabdallah/TrialMatchAI/src/Matcher/finetuned_reranker/"
reranker = Reranker(model_name, adapter_path, batch_size=20)

pairs = [['what is panda?', 'hi'], ['what is panda?', 'The giant panda (Ailuropoda melanoleuca), sometimes called a panda bear or simply panda, is a bear species endemic to China.']]*100
scores = reranker.rerank(pairs)
print(scores)


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import multiprocessing as mp

def get_inputs(pairs, tokenizer, prompt=None, max_length=1024):
    if prompt is None:
        prompt = "Analyze the medical condition described in query A and the statement in passage B. Determine if the statement is in accordance with the condition. Respond with 'Yes' if they are suitable and aligned or 'No' if otherwise."
    sep = "\n"
    prompt_inputs = tokenizer(prompt, return_tensors=None, add_special_tokens=False)['input_ids']
    sep_inputs = tokenizer(sep, return_tensors=None, add_special_tokens=False)['input_ids']
    inputs = []
    for query, passage in pairs:
        query_inputs = tokenizer(f'A: {query}', return_tensors=None, add_special_tokens=False, max_length=max_length * 3 // 4, truncation=True)
        passage_inputs = tokenizer(f'B: {passage}', return_tensors=None, add_special_tokens=False, max_length=max_length, truncation=True)
        item = tokenizer.prepare_for_model(
            [tokenizer.bos_token_id] + query_inputs['input_ids'],
            sep_inputs + passage_inputs['input_ids'],
            truncation='only_second',
            max_length=max_length,
            padding=False,
            return_attention_mask=False,
            return_token_type_ids=False,
            add_special_tokens=False
        )
        item['input_ids'] = item['input_ids'] + sep_inputs + prompt_inputs
        item['attention_mask'] = [1] * len(item['input_ids'])
        inputs.append(item)
    return tokenizer.pad(
        inputs,
        padding=True,
        max_length=max_length + len(sep_inputs) + len(prompt_inputs),
        pad_to_multiple_of=8,
        return_tensors='pt',
    )


def process_on_gpu(pairs, model_name, adapter_path, device_id, batch_size=16):
    # Load the tokenizer and model on the specific GPU
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)
    model = PeftModel.from_pretrained(model, adapter_path).to(f'cuda:{device_id}')
    model.eval()

    yes_loc = tokenizer('Yes', add_special_tokens=False)['input_ids'][0]

    # Process data in batches
    all_scores = []
    for i in range(0, len(pairs), batch_size):
        batch_pairs = pairs[i:i+batch_size]
        with torch.no_grad():
            inputs = get_inputs(batch_pairs, tokenizer).to(f'cuda:{device_id}')
            logits = model(**inputs, return_dict=True).logits[:, -1, yes_loc].view(-1).float()
            all_scores.extend(logits.cpu().tolist())

    return all_scores


def distribute_across_gpus(pairs, model_name, adapter_path, num_gpus, batch_size=16):
    # Split the pairs into chunks for each GPU
    chunk_size = len(pairs) // num_gpus
    pair_chunks = [pairs[i * chunk_size:(i + 1) * chunk_size] for i in range(num_gpus)]

    # Create a multiprocessing pool with one worker per GPU
    with mp.Pool(processes=num_gpus) as pool:
        results = pool.starmap(process_on_gpu, [(chunk, model_name, adapter_path, i, batch_size) for i, chunk in enumerate(pair_chunks)])

    # Flatten the list of results from all processes
    return [score for result in results for score in result]


# Usage example
if __name__ == "__main__":
    model_name = 'BAAI/bge-reranker-v2-gemma'
    adapter_path = "/home/mabdallah/TrialMatchAI/src/Matcher/finetuned_reranker/"
    
    pairs = [
        ['Patient with KRAS mutation', 'Patient without KRAS mutation'],
        ['what is panda?', 'The giant panda (Ailuropoda melanoleuca), sometimes called a panda bear or simply panda, is a bear species endemic to China.']
    ] * 1000

    # Use 2 GPUs for this example (adjust based on your available GPUs)
    num_gpus = 5
    batch_size = 16  # Adjust the batch size based on your available GPU memory
    scores = distribute_across_gpus(pairs, model_name, adapter_path, num_gpus, batch_size)
    print(scores)


In [ ]:
len(scores)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import multiprocessing as mp

def get_inputs(pairs, tokenizer, prompt=None, max_length=1024):
    if prompt is None:
        prompt = "Analyze the medical condition described in query A and the statement in passage B. Determine if the statement is in accordance with the condition. Respond with 'Yes' if they are suitable and aligned or 'No' if otherwise."
    sep = "\n"
    prompt_inputs = tokenizer(prompt, return_tensors=None, add_special_tokens=False)['input_ids']
    sep_inputs = tokenizer(sep, return_tensors=None, add_special_tokens=False)['input_ids']
    inputs = []
    for query, passage in pairs:
        query_inputs = tokenizer(f'A: {query}', return_tensors=None, add_special_tokens=False, max_length=max_length * 3 // 4, truncation=True)
        passage_inputs = tokenizer(f'B: {passage}', return_tensors=None, add_special_tokens=False, max_length=max_length, truncation=True)
        item = tokenizer.prepare_for_model(
            [tokenizer.bos_token_id] + query_inputs['input_ids'],
            sep_inputs + passage_inputs['input_ids'],
            truncation='only_second',
            max_length=max_length,
            padding=False,
            return_attention_mask=False,
            return_token_type_ids=False,
            add_special_tokens=False
        )
        item['input_ids'] = item['input_ids'] + sep_inputs + prompt_inputs
        item['attention_mask'] = [1] * len(item['input_ids'])
        inputs.append(item)
    return tokenizer.pad(
        inputs,
        padding=True,
        max_length=max_length + len(sep_inputs) + len(prompt_inputs),
        pad_to_multiple_of=8,
        return_tensors='pt',
    )


def process_on_gpu(pairs, model_name, adapter_path, device_id, batch_size=16):
    # Load the tokenizer and model on the specific GPU
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)
    model = PeftModel.from_pretrained(model, adapter_path).to(f'cuda:{device_id}')
    model.eval()

    yes_loc = tokenizer('Yes', add_special_tokens=False)['input_ids'][0]

    # Process data in batches
    all_scores = []
    for i in range(0, len(pairs), batch_size):
        batch_pairs = pairs[i:i+batch_size]
        with torch.no_grad():
            inputs = get_inputs(batch_pairs, tokenizer).to(f'cuda:{device_id}')
            logits = model(**inputs, return_dict=True).logits[:, -1, yes_loc].view(-1).float()
            all_scores.extend(logits.cpu().tolist())

    return all_scores


def distribute_across_gpus(pairs, model_name, adapter_path, device_ids, batch_size=16):
    # Split the pairs into chunks for each device
    num_gpus = len(device_ids)
    chunk_size = len(pairs) // num_gpus
    pair_chunks = [pairs[i * chunk_size:(i + 1) * chunk_size] for i in range(num_gpus)]

    # Create a multiprocessing pool with one worker per specified GPU
    with mp.Pool(processes=num_gpus) as pool:
        results = pool.starmap(process_on_gpu, [(chunk, model_name, adapter_path, device_id, batch_size) for chunk, device_id in zip(pair_chunks, device_ids)])

    # Flatten the list of results from all processes
    return [score for result in results for score in result]


# Usage example
if __name__ == "__main__":
    model_name = 'BAAI/bge-reranker-v2-gemma'
    adapter_path = "/home/mabdallah/TrialMatchAI/src/Matcher/finetuned_reranker/"
    
    pairs = [
        ['Patient with KRAS mutation', 'Patient without KRAS mutation'],
        ['what is panda?', 'The giant panda (Ailuropoda melanoleuca), sometimes called a panda bear or simply panda, is a bear species endemic to China.']
    ] * 10000

    # Specify the GPU devices you want to use
    device_ids = [1, 2 , 3, 4, 5]  # Use GPU 0 and GPU 1
    batch_size = 20  # Adjust the batch size based on your available GPU memory

    scores = distribute_across_gpus(pairs, model_name, adapter_path, device_ids, batch_size)
    print(scores)
